# 006 · Results-day move model (not started)

**The question.** How big will the price jump be when this company reports its results?

**The answer.** `move`: the absolute return from the close before the release to the close after the 8-K was filed, minus the market's (SPY). One row per release.

**The baseline to beat.** `prev4_mean`: the mean of the company's previous four moves (`n_prev` says how many it had; none for a company's first release).

**Measures.** rank correlation, mean and median absolute error, and for the chances the Brier score and log loss.

**Status.** Not started: headings and instructions only. The model is built, run and tuned in
this notebook in feature 006
([docs/plan.md](../docs/plan.md), issue in the roadmap).

### Rules for every model notebook

- **Spec first.** Before filling this notebook, run `/speckit-specify` for this feature with
  `SPECIFY_FEATURE_DIRECTORY=specs/006-results-move-model` on branch `006-results-move-model` (README, "Working
  together").
- **This notebook is where the model is built.** Every level is created, run and tuned here,
  with its tables and charts, start to finish by one command, so the researcher can read how the
  choice was made. Once the researcher is happy with the chosen model, its code moves into
  `src/sentinel/models/` (training, scoring, calibration) so that `sentinel update` scores every
  company each night without a notebook and the repository runs from a clone; this notebook then
  imports that code and must give the same numbers (docs/plan.md, "Notebooks and code").
- **Data.** Read only `data/exports/ml/results_move.parquet` through `sentinel.eda.load("results_move")`, which keeps
  the development rows (dated 2022 or earlier). Learn and test only on rows where this question's
  flag `eligible` is true and the answer is present (V2-26, V2-27). Never read `data/exports/reference/` (answers for 2023 on) and
  never compute anything for 2023 or later: that is the sealed holdout, tested once by feature 010
  (constitution IV).
- **Never use as features:** `move`, `close_after`, `filing_date`, `report_date`, `cik`, `as_of` and the flags; `prev4_mean` and `n_prev` only if declared as features in the spec.
- **Never a random train/test split.** Rows of one company over time are not independent, and a
  random split lets the model learn from the future. The only split is walk-forward by year:
- **Evaluation is fixed** (decision D60, `config/risk_params.yaml` v5, never edited): yearly
  walk-forward test years 2019, 2020, 2021, 2022; for each, train only on rows whose answer window
  ended before 1 January of that year (`sentinel.eda.purge_table` counts them); four levels:
  naive baseline → rule → classic model → LightGBM; choose the **simplest level within one
  standard error of the best**; report every test year, not only the average.
- **Show forecasts as chances of a move above set sizes (for example 5%, 10%, 20%), never one number** with their hit rates on the test years (D62), never as a bare
  number.
- **Reproducible:** sort rows by `cik`, `as_of` before modelling; fixed seeds (`SEED = 0`); log
  each run in MLflow with the git commit and the export date. Two runs must give identical
  predictions.
- **Private data:** outputs are stripped on commit (nbstripout); written conclusions quote
  aggregates only, never one company's prices or returns (Tiingo terms). Never write to `data/`.
- **Run it** start to finish under the memory cap:
  `systemd-run --user --scope -p MemoryMax=5G -p MemorySwapMax=0 uv run jupyter nbconvert --to notebook --execute --inplace notebooks/006-results-move-model.ipynb`

### Two kinds of test, and the final one is not in this notebook

1. **Test years 2019–2022 (here).** For each test year, train on the earlier years only (with
   the purge) and score that year. This is where the four levels are compared and **one is
   chosen**. Because the choice is made by looking at these results, they flatter the winner a
   little.
2. **The holdout: answers whose windows start in 2023 or later (not here, not in any notebook).**
   It answers the only question that matters in the end: does the model chosen here work on data
   nobody looked at while choosing it? That works only if it is used **once**.

**How it stays hidden.** The training file has no answers after 2022: the pipeline does not even
compute them (the label code stops at windows ending 31 December 2022). This notebook must not
read the private reference copies in `data/exports/reference/` (feature 002), which hold them.

**How the final test is run (feature 010, 16 November).**

1. This feature ends with the chosen model **frozen** in `src/sentinel/models/`: its settings,
   its training rows (to 2022), its seed, the git commit and the export date, logged in MLflow.
   The report (008) is written from the 2019–2022 results.
2. The researcher then runs **one guarded command** (built in 010). It checks that the models are
   frozen and that no holdout run exists yet, computes the 2023+ answers, scores the frozen
   models, and writes the results with a log entry. A second run is refused: the log is the proof
   that the test happened once.
3. The results go into the report as they came out. The model is not changed afterwards
   (constitution IV). Later refits (021) may use the 2023+ answers, as a logged step.

A command, not a notebook, because a notebook can be re-run with one click and nothing records
how often it was looked at.

### Make the model holdout-ready (so the one test runs without a hitch)

The holdout rows come from the same feature table as the training rows (`feature_snapshot`
from 2023 on: same columns, same order, same types, checked on 30 October) and their answers from
the same label code with a later end date. What can still break is the model itself, so:

1. **Select features by name, never by position**: save the list of feature columns with the
   model and use `rows[features]` when scoring.
2. **Every preprocessing step lives inside the model** (cut points, fill values, missing flags,
   scaling), fitted on the training rows only and saved with it, never recomputed on new rows.
3. **Save one object** (in MLflow): the fitted preprocessing and model, the feature list, the
   eligibility flag it uses (`{elig}`), the seed, the git commit and the export date.
4. **Dry run before 16 November**: load the saved model and score `latest.parquet` and the 2023+
   rows of `feature_snapshot.parquet` (features only, no answers): it must run, give one
   prediction per eligible row, no unexpected missing values, values in a plausible range. This
   looks at no outcome, so it does not spend the holdout.
5. **Scoring is a function** `predict(model, rows) -> predictions` in `src/sentinel/models/`,
   the same one used for the test years, the dry run, the holdout and the daily scores.

### What the EDA found for this model (notebooks/004-eda-results-move.ipynb, Findings)

- The baseline *ranks* releases (rank correlation 0.30–0.39 in 2019–2022) but its size is no
  more accurate than one constant for everyone (mean absolute error 0.046–0.052 against
  0.042–0.052). Judge by ranking and by the chances, not by average error alone.
- **Rule level suggestion:** the *median* of the previous four moves, shrunk toward the sector
  median when a company has few earlier moves.
- Volatility ranks the move as well as the company's own history (ρ ≈ 0.33): combine the two.
  Then drawdown, distance from the high, beta, loss-making, cash share, size.
- Model the log of the move or use a quantile loss: the tail is long (99% of moves below 31%).
- Financial features are mostly missing in 2010–2011: consider training from 2012.
- Check the test years with and without windows of 5 or more days.
- No foreign filers in this set (they do not file 8-K item 2.02).

**Data cleaning** (V2-33): the pipeline's cleaning is frozen at V2-32 (foreign market values,
share-count typos, insider value typos, placeholder prices, splits in the dilution answer, Tiingo
instruments; `docs/decisions.md`). Any further cleaning is part of **this model**: cut extreme
values at the 1st and 99th percentiles of the training rows, drop rows with a known doubt, add
"was missing" flags; write it here, explain why, and move it into `src/sentinel/models/` with the
model so the daily scoring cleans the same way. Do not rebuild the database for small fixes.

## 1. Setup and data

*To do:* Load the development rows, keep eligible rows with an answer, list the features to use (the EDA's advice below), and print rows and companies per test year.

## 2. Walk-forward folds

*To do:* Build the four folds (test years 2019–2022) with the purge; print train and test rows per fold.

## 3. Level 0: the naive baseline

*To do:* Score the baseline on each test year with the chosen measures.

## 4. Level 1: a rule

*To do:* A simple, explainable rule (see the EDA's suggestion below); fit only on each fold's training rows.

## 5. Level 2: a classic model

*To do:* A linear or logistic model: one feature of each redundant pair, ratios cut at their 1st and 99th percentiles (on training rows), missing values filled with the training median plus a 'was missing' flag, standardised.

## 6. Level 3: LightGBM

*To do:* Gradient-boosted trees with fixed seeds and settings chosen by time-ordered validation inside the training years only (never on the test year).

## 7. Comparison and choice

*To do:* One table: each level × each test year, the mean and its standard error; pick the simplest level within one standard error of the best.

## 8. Calibration and bands

*To do:* Turn the forecast into chances of a move above 5%, 10% and 20%; check each chance against the real share in each test year; five bands with their hit rates (D47, D62).

## 9. What the model uses

*To do:* Feature importance (LightGBM) or coefficients (classic); check that the strongest features make sense and none is a leak.

## 10. Findings

*To do:* In plain words: the chosen level and why, its skill per test year against the baseline, the bands and their hit rates, what to move into `src/sentinel/models/`, and open questions.